# 10 · Synthetic generalisation and holding-rate sweep

Loads the tables under `runs/jepa-replenishment/synthetic/` and states the
success rules beside them. In-distribution, unseen-SKU, and
unseen-disruption are separate tables, and the holding-rate sweep is a
fourth file. Each table is a bar chart of mean profit, with the
never-order, random-cover, and least-squares floors in grey and the
oracle or tuned ceiling in orange. The per-step rollout curve, the
capacity curve, and the data-scaling curve are line charts when those
files are present. The files are git-ignored. When the profit tables are
absent this notebook prints the published command that fills them and
does not invent a profit chart.

Set `JEPA_SYNTHETIC_DIR` to read a different directory. The published
log, fit, and tuning studies are in the root README section
[Latent replenishment](../README.md#latent-replenishment).


In [ ]:
import os
from pathlib import Path

import matplotlib

matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt

while not (Path.cwd() / "pyproject.toml").exists():
    os.chdir("..")

from src.jepa.commands import EVAL_COMMAND
from src.jepa.eval import is_profit_column

directory = Path(os.environ.get("JEPA_SYNTHETIC_DIR", "runs/jepa-replenishment/synthetic"))
scaling_dir = Path(os.environ.get("JEPA_SCALING_DIR", directory.parent / "scaling"))
PROTOCOL_FILES = (
    "in_distribution.parquet",
    "unseen_sku.parquet",
    "unseen_disruption.parquet",
)
TABLES_PRESENT = all((directory / name).is_file() for name in PROTOCOL_FILES)
SWEEP_PRESENT = (directory / "holding_rate_sweep.parquet").is_file()
SWEEP_REFUSED = (directory / "holding_rate_sweep_refused.json").is_file()
if not TABLES_PRESENT:
    print(f"{directory} has no synthetic tables.")
    print("The published command that fills them is:")
    print(EVAL_COMMAND)
    refused = directory / "profit_table_refused.json"
    if refused.is_file():
        print(refused.read_text())

FLOOR_ARMS = {"never_order", "random_cover", "least_squares"}
CEILING_ARMS = {"oracle", "ceiling"}


def profit_means(frame):
    return [
        (str(column), float(frame[column].mean()))
        for column in frame.columns
        if is_profit_column(str(column))
    ]


def print_means(frame):
    for name, value in profit_means(frame):
        print(f"{name} {value:.3f}")


def bar_means(title, frame):
    pairs = profit_means(frame)
    if not pairs:
        return
    _fig, ax = plt.subplots(figsize=(8, 3.6))
    names = [name for name, _value in pairs]
    values = [value for _name, value in pairs]
    colors = []
    for name in names:
        if name in FLOOR_ARMS:
            colors.append("#bdbdbd")
        elif name in CEILING_ARMS:
            colors.append("#f4a261")
        elif name in {"latent", "anchor"}:
            colors.append("#2a9d8f")
        else:
            colors.append("#457b9d")
    ax.bar(names, values, color=colors)
    ax.set_title(title)
    ax.set_ylabel("mean net_profit / initial_cash")
    ax.tick_params(axis="x", rotation=40, labelsize=8)
    _fig.tight_layout()


def line_curve(title, xlabel, ylabel, series):
    _fig, ax = plt.subplots(figsize=(7, 3.6))
    for label, xs, ys in series:
        ax.plot(xs, ys, marker="o", label=label)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend()
    _fig.tight_layout()


## In-distribution

Training SKUs, default disruption, tuner holdout, `holding_rate` 0.01.
The rule is printed with the table:
the learned arm beats the tuned anchor, and the bootstrap interval of that
paired difference excludes zero. A tie is not a pass. The table's floors are
never-order, random-cover, and least-squares-in-planner; the oracle model
in the planner is the ceiling. Order volume, service level, stockout rate,
and realised fill rate sit beside profit, with held-out rollout R² on the
same rows when the publish gate has opened. The unseen protocols below are
not pooled into this table.


In [ ]:
import pandas as pd

from src.jepa.eval import IN_DISTRIBUTION_RULE, in_distribution_passes

if TABLES_PRESENT:
    in_dist = pd.read_parquet(directory / "in_distribution.parquet")
    print(IN_DISTRIBUTION_RULE)
    print_means(in_dist)
    bar_means("In-distribution mean profit", in_dist)
    if {"latent", "anchor"} <= set(in_dist.columns):
        print(
            "in_distribution_passes",
            in_distribution_passes(in_dist["latent"], in_dist["anchor"]),
        )
    else:
        print(
            "in_distribution_passes unavailable; "
            "latent or anchor was dropped from the parquet"
        )
    warnings = directory / "comparison_warnings.json"
    if warnings.is_file():
        print(warnings.read_text())


## Unseen SKU

Evaluation SKUs only, default disruption, 32 seeds from `15_000_000`.
Same floors, oracle ceiling, and operational columns as the in-distribution
table. This is not a seed holdout of the training catalog.


In [ ]:
from src.jepa.eval import UNSEEN_SKU_SEED_OFFSET

if TABLES_PRESENT:
    unseen = pd.read_parquet(directory / "unseen_sku.parquet")
    print("protocol unseen_sku", UNSEEN_SKU_SEED_OFFSET)
    print_means(unseen)
    bar_means("Unseen SKU mean profit", unseen)


## Unseen disruption

Training SKUs, the harsher disruption regime the log never samples, and
its own seed block from `16_000_000`. Same columns as the other two
protocols. The SKUs match in-distribution. The regime and the seeds do not.


In [ ]:
from src.jepa.eval import UNSEEN_DISRUPTION_SEED_OFFSET

if TABLES_PRESENT:
    shocked = pd.read_parquet(directory / "unseen_disruption.parquet")
    print("protocol unseen_disruption", UNSEEN_DISRUPTION_SEED_OFFSET)
    print_means(shocked)
    bar_means("Unseen disruption mean profit", shocked)


## Holding-rate sweep

Same holdout seeds at each rate `0.005, 0.01, 0.02, 0.05, 0.1`, weights
from the 0.01 fit, no new log. At each rate the frozen planner is scored
against the `OrderUpToPolicy` tuned at that rate. The file is written only
when that planner's order sequence differs across the rates; a no-op
protocol raises and leaves no curve.


In [ ]:
from src.jepa.eval import HOLDING_RATE_SWEEP_RULE

if SWEEP_REFUSED:
    print(HOLDING_RATE_SWEEP_RULE)
    print("The holding-rate sweep was refused.")
    print((directory / "holding_rate_sweep_refused.json").read_text())
elif SWEEP_PRESENT:
    sweep = pd.read_parquet(directory / "holding_rate_sweep.parquet")
    print(HOLDING_RATE_SWEEP_RULE)
    print(
        "sweep_rates",
        tuple(float(rate) for rate in sorted(sweep["holding_rate"].unique())),
    )
    for rate in sorted(sweep["holding_rate"].unique()):
        group = sweep[sweep["holding_rate"] == rate]
        print("holding_rate", float(rate))
        print_means(group)
    series = []
    for arm in ("latent", "ceiling"):
        if arm not in sweep.columns:
            continue
        grouped = sweep.groupby("holding_rate")[arm].mean()
        series.append((arm, list(grouped.index), list(grouped.values)))
    if series:
        line_curve(
            "Holding-rate sweep",
            "holding_rate",
            "mean net_profit / initial_cash",
            series,
        )
elif TABLES_PRESENT:
    print(HOLDING_RATE_SWEEP_RULE)
    print("synthetic/holding_rate_sweep.parquet is not in this run")


## Rollout accuracy and scaling curves

The per-step curve is `rollout_accuracy.json` beside the tables. It is
plotted whether or not the profit gate opened. Capacity and data scaling
are `capacity.json` and `data.json` under the sibling `scaling/` directory
(or `JEPA_SCALING_DIR`). A missing file is skipped. No number is filled in.

In [ ]:
import json

rollout_path = directory / "rollout_accuracy.json"
if rollout_path.is_file():
    rollout = json.loads(rollout_path.read_text())
    line_curve(
        "Held-out rollout R²",
        "step",
        "R²",
        [
            (name, rollout["steps"], rollout[name])
            for name in ("latent", "direct", "ols")
            if name in rollout
        ],
    )
    for name in ("latent", "direct", "ols"):
        values = [value for value in rollout.get(name, []) if value is not None]
        if values:
            print(f"rollout_r2_{name}_mean {sum(values) / len(values):.6f}")


def _scaling_series(points, x_key):
    series = []
    for label, y_key in (
        ("latent", "latent_mean"),
        ("direct", "direct_mean"),
        ("ols", "ols_mean"),
    ):
        xs = [point[x_key] for point in points if y_key in point]
        ys = [point[y_key] for point in points if y_key in point]
        if xs:
            series.append((label, xs, ys))
    return series


capacity_path = scaling_dir / "capacity.json"
if capacity_path.is_file():
    capacity = json.loads(capacity_path.read_text())
    line_curve(
        "Model capacity",
        "parameters",
        "mean rollout R²",
        _scaling_series(capacity["points"], "parameters"),
    )
    print("capacity_points", len(capacity["points"]))

data_path = scaling_dir / "data.json"
if data_path.is_file():
    data_curve = json.loads(data_path.read_text())
    line_curve(
        "Data scaling",
        "episodes",
        "mean rollout R²",
        _scaling_series(data_curve["points"], "episodes"),
    )
    print("data_points", len(data_curve["points"]))

print("figures", len(plt.get_fignums()))
